# Análise Exploratória e Limpeza de Dados (EDA)
Vamos limpar os metadados brutos do dataset MIQR-CC, alinhar as categorias com as 4 exigidas pelo enunciado e realizar a separação de treino, validação e teste.

## 0. Imports

In [13]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn


## 1. Carregamento e Filtragem Base
Filtragem das imagens rejeitadas pelos médicos (coluna `Keep` == 'Discard') e exclusão das imagens sem patologia identificada (`Unlabelled`).

In [8]:
# Caminho relativo limpo e padrão, partindo da pasta 'notebooks/'
csv_path = os.path.join("..", "data", "MIQR-CC-Dataset", "metadata.csv")

# Carregar o dataset
df = pd.read_csv(csv_path)

# Filtrar: Manter as marcadas como 'Keep' e descartar 'Unlabelled'
df_limpo = df[(df['Keep'] == 'Keep') & (df['Label'] != 'Unlabelled')].copy()

print(f"Total de imagens originais: {len(df)}")
print(f"Total de imagens úteis para treino: {len(df_limpo)}")

Total de imagens originais: 19317
Total de imagens úteis para treino: 1568


## 2. Mapeamento de Classes
O dataset divide as estenoses em malignas e benignas. Vamos mapear os dados para as 4 classes obrigatórias: **Biliary_Leaks, Lithiasis, Stricture e Normal**.

In [9]:
def mapear_classes(label):
    if label == 'Biliary Leaks':
        return 'Biliary_Leaks' # Adicionar o underscore
    elif label in ['Malignant Stricture', 'Benign Stricture']:
        return 'Stricture' # Agrupar as estenoses
    else:
        return label

# Criar uma nova coluna com os nomes padronizados
df_limpo['Label_Mapped'] = df_limpo['Label'].apply(mapear_classes)

## 3. Distribuição das Classes (Balanceamento)
Verificação do rácio de patologias para posterior cálculo dos Class Weights.

In [10]:
print("--- Distribuição das Classes ---")
contagem = df_limpo['Label_Mapped'].value_counts()
percentagem = df_limpo['Label_Mapped'].value_counts(normalize=True) * 100

for classe, qtd in contagem.items():
    print(f"Classe: {classe:<15} | Quantidade: {qtd:<5} | Proporção: {percentagem[classe]:.2f}%")

--- Distribuição das Classes ---
Classe: Lithiasis       | Quantidade: 726   | Proporção: 46.30%
Classe: Stricture       | Quantidade: 392   | Proporção: 25.00%
Classe: Normal          | Quantidade: 299   | Proporção: 19.07%
Classe: Biliary_Leaks   | Quantidade: 151   | Proporção: 9.63%


## 4. Divisão Estratificada (Train, Val, Test)
Divisão dos dados em **70% Treino, 15% Validação e 15% Teste**, mantendo a representatividade das classes minoritárias com *Stratified Sampling*.

In [ ]:
SEMENTE = 42 

# 1º Split: Treino (70%) e Resto (30%)
train_df, rest_df = train_test_split(
    df_limpo, test_size=0.30, random_state=SEMENTE, stratify=df_limpo['Label_Mapped']
)

# 2º Split: Validação (15%) e Teste (15%)
val_df, test_df = train_test_split(
    rest_df, test_size=0.50, random_state=SEMENTE, stratify=rest_df['Label_Mapped']
)

print(f"Subconjunto de Treino:    {len(train_df)} imagens")
print(f"Subconjunto de Validação: {len(val_df)} imagens")
print(f"Subconjunto de Teste:     {len(test_df)} imagens")

# Guardar em CSVs para serem lidos pelo data_setup.py no momento do treino
pasta_splits = os.path.join("..", "data", "MIQR-CC-Dataset", "splits")
os.makedirs(pasta_splits, exist_ok=True)

train_df.to_csv(os.path.join(pasta_splits, "train.csv"), index=False)
val_df.to_csv(os.path.join(pasta_splits, "val.csv"), index=False)
test_df.to_csv(os.path.join(pasta_splits, "test.csv"), index=False)
print(f"\nSucesso! Ficheiros guardados em: {pasta_splits}")

Subconjunto de Treino:    1097 imagens
Subconjunto de Validação: 235 imagens
Subconjunto de Teste:     236 imagens

Sucesso! Ficheiros guardados em: ..\data\MIQR-CC-Dataset\splits


## 5. Estratégias de Desequilíbrio: Class Weights
Como detetámos um forte desequilíbrio entre as classes (ex: quase 5x mais imagens de *Lithiasis* do que de *Biliary_Leaks*), precisamos de penalizar mais o modelo quando este erra nas classes minoritárias. 

Vamos utilizar a biblioteca `scikit-learn` para calcular os pesos balanceados rigorosos.

In [14]:

# 1. Obter as labels exatas que temos no dataset limpo
labels = df_limpo['Label_Mapped'].values
classes_unicas = np.unique(labels)

# 2. Calcular os pesos balanceados automaticamente
pesos = compute_class_weight(class_weight='balanced', classes=classes_unicas, y=labels)

# 3. Criar um dicionário para visualizar melhor
pesos_dict = dict(zip(classes_unicas, pesos))

print("--- Pesos Calculados por Classe ---")
for classe, peso in pesos_dict.items():
    print(f"{classe:<15}: {peso:.4f}")

--- Pesos Calculados por Classe ---
Biliary_Leaks  : 2.5960
Lithiasis      : 0.5399
Normal         : 1.3110
Stricture      : 1.0000


### 5.1. Preparação para a Função de Perda (CrossEntropyLoss)
No PyTorch, para passarmos estes pesos para a função de erro, precisamos que eles estejam num Tensor (vetor matemático) e **exatamente na mesma ordem** que definimos no nosso dicionário de mapeamento no ficheiro `data_setup.py` (0: Biliary_Leaks, 1: Lithiasis, 2: Stricture, 3: Normal).

In [15]:

# Ordem exigida pelo nosso data_setup.py
ordem_classes = ['Biliary_Leaks', 'Lithiasis', 'Stricture', 'Normal']

# Organizar os pesos nessa ordem
pesos_ordenados = [pesos_dict[c] for c in ordem_classes]

# Converter para um Tensor do PyTorch
tensor_pesos = torch.tensor(pesos_ordenados, dtype=torch.float)
print(f"\nTensor de Pesos final para o PyTorch: \n{tensor_pesos}")

# Como será injetado na Tarefa 4:
criterio = nn.CrossEntropyLoss(weight=tensor_pesos)
print("\nPreparação concluída! O critério já está configurado para lidar com o desequilíbrio.")


Tensor de Pesos final para o PyTorch: 
tensor([2.5960, 0.5399, 1.0000, 1.3110])

Preparação concluída! O critério já está configurado para lidar com o desequilíbrio.
